# Colab Pro: resume S09 and export a recall trial

This notebook reuses the frozen S09 candidate bundle and model scores. It exports one ZIP with one JSON at its root for manual submission. Default trial: 40 documents, 80 chunks, 4 chunks/document. It does not add ICD/NER or expand the 200-candidate pool; relevance improvement is unmeasured.

**Fast path:** completed worker outputs require only CPU. **GPU path:** set RUN_MISSING_RERANK=True to finish missing scores with the original pinned model, sequentially on this one Colab GPU. Existing matching query scores are reused. Do not run another writer against the same backup directory.

Required: matching stage9_rerank_colab.zip, completed workers or permission to compute missing scores, and official query/links Parquet files (or Data.zip). The older stage5 bundles cannot replace the S09 bundle. Edit paths in cell 1; missing inputs stop with an actionable message. Existing production outputs remain untouched. Run all cells in order in a fresh runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, zipfile, tempfile, shutil, sys, subprocess, os
ROOT = Path('/content/drive/MyDrive/AIGuru')
BUNDLE = ROOT / 'stage9_rerank_colab.zip'
WORKERS = ROOT / 'stage9_rerank_workers'
DATA_ZIP = ROOT / 'Data.zip'
QUERY_FILE = ROOT / 'Data/official/ViBioMIR/query.parquet'
LINKS_FILE = ROOT / 'Data/official/ViBioMIR/links_corpus_vi_en_zh.parquet'
LINKS_ID_COLUMN = 'id'  # Change only if the official file uses another ID column.
ARTIFACTS_ZIP = ROOT / 'artifacts.zip'  # optional source of the S09 bundle/worker files
RUN_MISSING_RERANK = False  # True requires CUDA; missing scores may take substantial time
RERANK_BACKUP = ROOT / 'stage9_submission_resume'
EXPECTED_LINKS_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LIMITS = dict(max_docs=40, max_chunks=80, max_chunks_per_doc=4, overlap_iou_threshold=0.8)
EXPECTED_QUERIES = 1200
OUTPUT_PARENT = ROOT / 'submission_trials'
print('Root files:')
assert ROOT.is_dir(), f'Drive directory missing: {ROOT}'
for p in ROOT.iterdir():
    print(p.name)
print('Bundle:', BUNDLE, 'exists:', BUNDLE.is_file())
print('Worker folder:', WORKERS, 'exists:', WORKERS.is_dir())


In [ ]:
# Extract only a nested S09 bundle if present, not the full corpus archive.
if not BUNDLE.is_file() and ARTIFACTS_ZIP.is_file():
    with zipfile.ZipFile(ARTIFACTS_ZIP) as archive:
        matches = [n for n in archive.namelist() if Path(n).name == 'stage9_rerank_colab.zip']
        assert len(matches) <= 1, 'Multiple S09 bundles in artifacts.zip; select the matching bundle explicitly.'
        if matches:
            local_bundle = Path(tempfile.mkdtemp(prefix='s09_bundle_', dir='/content')) / 'stage9_rerank_colab.zip'
            with archive.open(matches[0]) as src, local_bundle.open('wb') as dst:
                shutil.copyfileobj(src, dst)
            BUNDLE = local_bundle
assert BUNDLE.is_file(), f'Missing S09 bundle: {BUNDLE}. Upload the matching stage9_rerank_colab.zip from the completed S09 run; a Stage-5 ZIP is not suitable.'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyarrow==22.0.0'], check=True)
PROJECT = Path(tempfile.mkdtemp(prefix='vibiomir_trial_', dir='/content'))
def digest(data):
    return hashlib.sha256(data).hexdigest()
def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
with zipfile.ZipFile(BUNDLE) as z:
    meta = json.loads(z.read('BUNDLE.json'))
    assert digest(canonical({k:v for k,v in meta.items() if k != 'signature'}).encode()) == meta['signature']
    assert len(z.namelist()) == len(set(z.namelist()))
    assert set(z.namelist()) == set(meta['files']) | {'BUNDLE.json'}
    assert shutil.disk_usage('/content').free > sum(i.file_size for i in z.infolist()) + 1024**3
    for name, expected in meta['files'].items():
        target = (PROJECT / name).resolve()
        assert PROJECT.resolve() in target.parents, 'Unsafe bundle path'
        payload = z.read(name)
        assert digest(payload) == expected, f'Bundle checksum mismatch: {name}'
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
# Use exactly the frozen source that created the scores.
assert not any(n == 'vibiomir' or n.startswith('vibiomir.') for n in sys.modules), 'Restart runtime to avoid mixing source versions.'
sys.path.insert(0, str(PROJECT))
from vibiomir.stage9_pipeline import prepared, load_packet, verify_scores, _worker_identity, import_worker
from vibiomir.stage9_core import select
from vibiomir.retrieval_contracts import checked, queries
from vibiomir.io import file_hash, text_hash, canonical_json
state = prepared(PROJECT)
assert state['identity']['scope'] == 'all_queries', 'A pilot bundle cannot produce a full submission.'
assert len(state['packets']) == EXPECTED_QUERIES
assert all(type(LIMITS[k]) is int and LIMITS[k] > 0 for k in ('max_docs','max_chunks','max_chunks_per_doc'))
assert 0 < LIMITS['overlap_iou_threshold'] <= 1
print('Verified prepared queries:', len(state['packets']))


In [ ]:
import pyarrow.parquet as pq
def official_file(path, basenames):
    if path.is_file():
        return path
    assert DATA_ZIP.is_file(), f'Missing {path}. Set the path or provide Data.zip.'
    with zipfile.ZipFile(DATA_ZIP) as z:
        matches = [n for n in z.namelist() if Path(n).name in basenames and 'official' in Path(n).parts]
        assert len(matches) == 1, f'Expected one official file, found {matches}. Extract the intended file and set its path.'
        target = PROJECT / 'official' / Path(matches[0]).name
        target.parent.mkdir(exist_ok=True)
        with z.open(matches[0]) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)
        return target
query_path = official_file(QUERY_FILE, {'query.parquet'})
links_path = official_file(LINKS_FILE, {'links_corpus_vi_en_zh.parquet', 'links_corpus.parquet'})
official_queries = queries(query_path)
assert file_hash(query_path) == state['identity']['query_file_sha256'], 'Query snapshot differs from frozen run.'
assert [q['id'] for q in official_queries] == [p['id'] for p in state['packets']]
assert file_hash(links_path) == EXPECTED_LINKS_SHA256, 'Official links differ from the verified baseline snapshot; verify the file before continuing.'
pf = pq.ParquetFile(links_path)
assert LINKS_ID_COLUMN in pf.schema_arrow.names, f'Set LINKS_ID_COLUMN. Available: {pf.schema_arrow.names}'
allowed = set()
for batch in pf.iter_batches(columns=[LINKS_ID_COLUMN]):
    for value in batch.column(0).to_pylist():
        assert type(value) is int, 'Official IDs must be integers.'
        allowed.add(value)
assert allowed


In [ ]:
# Search only expected worker locations; never unpack the full corpus.
restored_workers = PROJECT / 'restored_workers'
restored_workers.mkdir()
worker_roots, worker_manifests = {}, {}
def locate_worker(i):
    for parent in (RERANK_BACKUP, WORKERS):
        folder = parent / f'worker_{i}'
        if (folder / 'manifest.json').is_file():
            return folder
    for archive in (WORKERS / f'stage9_worker_{i}.zip', ROOT / f'stage9_worker_{i}.zip'):
        if archive.is_file():
            import_worker(PROJECT, archive, restored_workers)
            return restored_workers / f'worker_{i}'
    if ARTIFACTS_ZIP.is_file():
        with zipfile.ZipFile(ARTIFACTS_ZIP) as z:
            matches = [n for n in z.namelist() if Path(n).name == f'stage9_worker_{i}.zip']
            assert len(matches) <= 1, 'Ambiguous worker archives; extract the matching one and set WORKERS.'
            if matches:
                archive = PROJECT / f'stage9_worker_{i}.zip'
                with z.open(matches[0]) as src, archive.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                import_worker(PROJECT, archive, restored_workers)
                return restored_workers / f'worker_{i}'
            prefixes = [n[:-len('manifest.json')] for n in z.namelist()
                        if n.endswith(f'stage9_rerank_workers/worker_{i}/manifest.json')]
            assert len(prefixes) <= 1, 'Ambiguous worker folders in artifacts.zip.'
            if prefixes:
                prefix = prefixes[0]
                folder = restored_workers / f'worker_{i}'
                folder.mkdir(exist_ok=True)
                names = ['manifest.json'] + [f"query-{p['id']}.json" for p in state['packets'] if p['worker_id'] == i]
                for name in names:
                    if prefix + name in z.namelist():
                        with z.open(prefix + name) as src, (folder / name).open('wb') as dst:
                            shutil.copyfileobj(src, dst)
                return folder
    return None

if RUN_MISSING_RERANK:
    os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(PROJECT / 'requirements-stage9-colab.txt')], check=True)
    import torch
    assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun from a fresh runtime.'
    from vibiomir.stage9_reranker import recommended_runtime
    preset = recommended_runtime(torch.cuda.get_device_properties(0).total_memory / 1024**3)
    print('GPU:', torch.cuda.get_device_name(0), 'budgets:', preset)

for i in range(state['identity']['settings']['workers']):
    folder = locate_worker(i)
    ident = _worker_identity(state, i)
    expected_ids = [p['id'] for p in state['packets'] if p['worker_id'] == i]
    manifest = json.loads((folder / 'manifest.json').read_text()) if folder else None
    if manifest:
        assert manifest['identity'] == ident and manifest['signature'] == text_hash(canonical_json(ident)), 'Worker belongs to another run.'
    complete = manifest is not None and manifest['state'] == 'complete' and manifest.get('completed_query_ids') == expected_ids
    files_present = folder is not None and all((folder / f'query-{q}.json').is_file() for q in expected_ids)
    if not (complete and files_present):
        assert RUN_MISSING_RERANK, f'Worker {i} missing/incomplete. Supply matching completed worker ZIPs, or enable RUN_MISSING_RERANK on CUDA.'
        local = PROJECT / 'resume_workers' / f'worker_{i}'
        local.mkdir(parents=True, exist_ok=True)
        if folder:
            for entry in state['packets']:
                if entry['worker_id'] != i: continue
                path = folder / f"query-{entry['id']}.json"
                if path.is_file():
                    packet = load_packet(PROJECT, entry)
                    verify_scores(checked(path), packet, entry, state, text_hash(canonical_json(ident)))
                    shutil.copy2(path, local / path.name)
        command = [sys.executable, '-u', '-m', 'vibiomir.stage9_pipeline', 'rerank-worker',
                   '--prepared-dir', str(PROJECT), '--output-dir', str(local), '--worker-id', str(i),
                   '--backup-dir', str(RERANK_BACKUP / f'worker_{i}'),
                   '--batch-size', str(preset['batch_size']), '--padded-tokens', str(preset['padded_tokens'])]
        subprocess.run(command, cwd=PROJECT, check=True)
        folder = local
        manifest = json.loads((folder / 'manifest.json').read_text())
    assert manifest['identity'] == ident and manifest['signature'] == text_hash(canonical_json(ident))
    assert manifest['state'] == 'complete' and manifest['completed_query_ids'] == expected_ids
    worker_roots[i], worker_manifests[i] = folder, manifest
print('Workers ready; every query score will be verified before export.')


In [ ]:
from collections import Counter
predictions, baseline, diagnostics, input_scores = [], [], [], {}
for number, entry in enumerate(state['packets'], 1):
    packet = load_packet(PROJECT, entry)
    assert packet['id'] == official_queries[number-1]['id'] and packet['query'] == official_queries[number-1]['query']
    worker_id = entry['worker_id']
    score_path = worker_roots[worker_id] / f"query-{entry['id']}.json"
    record = checked(score_path)
    scores = [r['score'] for r in verify_scores(record, packet, entry, state, worker_manifests[worker_id]['signature'])]
    input_scores[str(entry['id'])] = file_hash(score_path)
    candidates = packet['candidates']
    for item in candidates:
        assert type(item['doc_id']) is int and item['doc_id'] in allowed, 'Ineligible document in prepared pool'
        assert item['chunk_text'].strip() and text_hash(item['chunk_text']) == item['text_sha256']
        assert item['region_end'] - item['region_start'] == len(item['chunk_text'])
        assert item['source_end'] - item['source_start'] == len(item['chunk_text'])
    original, old_review = select(candidates, scores, state['identity']['settings']['selection'])
    trial, review = select(candidates, scores, LIMITS)
    assert len(trial['relevant_docs']) == len(set(trial['relevant_docs'])) <= LIMITS['max_docs']
    assert len(trial['relevant_chunks']) <= LIMITS['max_chunks']
    assert all(c['doc_id'] in trial['relevant_docs'] for c in trial['relevant_chunks'])
    assert all(n <= LIMITS['max_chunks_per_doc'] for n in Counter(c['doc_id'] for c in trial['relevant_chunks']).values())
    predictions.append({'id': packet['id'], **trial})
    baseline.append({'id': packet['id'], **original})
    diagnostics.append(dict(id=packet['id'], pool_documents=len({c['doc_id'] for c in candidates}),
        old_documents=len(original['relevant_docs']), new_documents=len(trial['relevant_docs']),
        old_chunks=len(original['relevant_chunks']), new_chunks=len(trial['relevant_chunks']),
        baseline_rejections=dict(Counter(d['reason'] for d in old_review['decisions'])),
        new_docs_without_chunks=len(set(trial['relevant_docs']) - {c['doc_id'] for c in trial['relevant_chunks']}),
        trial_rejections=dict(Counter(d['reason'] for d in review['decisions']))))
    if number % 100 == 0:
        print(f'Checked and selected {number}/{len(state["packets"])} queries')
assert [p['id'] for p in predictions] == [q['id'] for q in official_queries]
changed = sum(a != b for a,b in zip(predictions, baseline))
assert changed, 'No predictions changed. Inspect candidate diversity before creating another submission.'
print('Changed queries:', changed)
print('Baseline docs/chunks:', sum(len(p['relevant_docs']) for p in baseline), sum(len(p['relevant_chunks']) for p in baseline))
print('Trial docs/chunks:', sum(len(p['relevant_docs']) for p in predictions), sum(len(p['relevant_chunks']) for p in predictions))


The checks above bind unchanged evidence to signed prepared packets and verified worker scores. They do not re-scan the original full Stage-5 source store or measure relevance. Official ID checks use your supplied official snapshot. Preserve the baseline; submit the new ZIP manually only after reviewing counts.


In [ ]:
from datetime import datetime, timezone
run = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out = OUTPUT_PARENT / (f"recall_{LIMITS['max_docs']}_{LIMITS['max_chunks']}_" + run)
out.mkdir(parents=True, exist_ok=False)
payload = (json.dumps(predictions, ensure_ascii=False, indent=2) + '\n').encode('utf-8')
(out / 'submission.json').write_bytes(payload)
with zipfile.ZipFile(out / 'submission.zip', 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.writestr('submission.json', payload)
with zipfile.ZipFile(out / 'submission.zip') as z:
    assert z.namelist() == ['submission.json'] and z.read('submission.json') == payload
assert json.loads(payload) == predictions
baseline_payload = (json.dumps(baseline, ensure_ascii=False, indent=2) + '\n').encode('utf-8')
(out / 'baseline_reconstructed.json').write_bytes(baseline_payload)
report = dict(operation='stage9-reselection-trial', limits=LIMITS,
    baseline_reconstructed_sha256=digest(baseline_payload),
    original_limits=state['identity']['settings']['selection'],
    prepared_signature=state['signature'], bundle_sha256=file_hash(BUNDLE),
    official_query_sha256=file_hash(query_path), official_links_sha256=file_hash(links_path),
    worker_query_sha256=input_scores, submission_sha256=digest(payload),
    zip_sha256=file_hash(out / 'submission.zip'), queries=len(predictions),
    changed_queries=changed, diagnostics=diagnostics,
    validation_scope='Signed prepared evidence, worker score identity, official IDs, coverage and ZIP/schema; original source store not rescanned.',
    gpu_completion_enabled=RUN_MISSING_RERANK, relevance_measured=False)
(out / 'REPORT.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Submission ZIP:', out / 'submission.zip')
print('Keep REPORT.json for comparison; do not include it in the submitted ZIP.')
from google.colab import files
files.download(str(out / 'submission.zip'))
